In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use("default")
sns.set_theme()


In [ ]:
# Load the World Development Indicators extract (replace filename if needed)
df = pd.read_csv("500eccd6-936f-4340-b991-d197797ee9c9_Data.csv")

df.head()


In [ ]:
# Reshape from wide (years as columns) to long format
df_long = df.melt(
    id_vars=["Country Name", "Country Code", "Series Name"],
    var_name="Year",
    value_name="Value"
)

df_long.head()


In [ ]:
# Extract the 4-digit year from values like "2020 [YR2020]"
df_long["Year"] = df_long["Year"].str.extract(r'(\d{4})')
df_long = df_long.dropna(subset=["Year"])
df_long["Year"] = df_long["Year"].astype(int)

# Convert numeric values and coerce non-numeric like ".." to NaN
df_long["Value"] = pd.to_numeric(df_long["Value"], errors="coerce")

df_long.head()


In [ ]:
selected_indicators = [
    "Life expectancy at birth, total (years)",
    "Mortality rate, adult, female (per 1,000 female adults)",
    "Mortality rate, adult, male (per 1,000 male adults)",
    "Mortality rate, infant (per 1,000 live births)",
    "Mortality rate, under-5 (per 1,000 live births)",
    "Diabetes prevalence (% of population ages 20 to 79)",
    "Physicians (per 1,000 people)",
    "Hospital beds (per 1,000 people)",
    "Current health expenditure (% of GDP)",
    "GDP per capita (current US$)",
    "School enrollment, secondary (% gross)"
]

df_long = df_long[df_long["Series Name"].isin(selected_indicators)]
df_long.head()


In [ ]:
# Pivot so each indicator becomes its own column
df_final = df_long.pivot_table(
    index=["Country Name", "Country Code", "Year"],
    columns="Series Name",
    values="Value",
    aggfunc="mean"
).reset_index()

df_final.head()


In [ ]:
# Drop rows with any missing values in the selected indicators
df_final = df_final.dropna()

df_final.head(), df_final.shape


In [ ]:
df_final.info()


In [ ]:
df_final.describe().T


In [ ]:
df_final.isnull().sum()


In [ ]:
numerical_cols = df_final.select_dtypes(include=["float64", "int64"]).columns
numerical_cols


In [ ]:
# Histograms for each numerical column
for col in numerical_cols:
    plt.figure(figsize=(6, 4))
    sns.histplot(df_final[col], kde=True)
    plt.title(f"Histogram of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()


In [ ]:
# Box plots for each numerical column
for col in numerical_cols:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=df_final[col])
    plt.title(f"Box Plot of {col}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()


In [ ]:
# Correlation heatmap (excluding ID-like columns)
corr = df_final.drop(columns=["Country Name", "Country Code", "Year"]).corr()

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)
plt.title("Correlation Heatmap of Indicators")
plt.tight_layout()
plt.show()


In [ ]:
# Pairplot of selected key indicators
pairplot_cols = [
    "Life expectancy at birth, total (years)",
    "GDP per capita (current US$)",
    "Current health expenditure (% of GDP)",
    "Mortality rate, adult, male (per 1,000 male adults)",
    "School enrollment, secondary (% gross)"
]

sns.pairplot(df_final[pairplot_cols], diag_kind="hist")
plt.suptitle("Pairplot of Selected Indicators", y=1.02)
plt.show()


In [ ]:
target_col = "Life expectancy at birth, total (years)"

X = df_final.drop(
    columns=[
        "Country Name",
        "Country Code",
        "Year",
        target_col
    ]
)

y = df_final[target_col]

X.shape, y.shape


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print("Linear Regression Performance:")
print("MAE:", mae_lr)
print("RMSE:", rmse_lr)
print("R²:", r2_lr)


In [ ]:
plt.figure(figsize=(8, 4))
indices = range(len(y_test))

plt.bar(indices, y_test, label="Actual", alpha=0.7)
plt.bar(indices, y_pred_lr, label="Predicted", alpha=0.7)

plt.title("Linear Regression: Actual vs Predicted Life Expectancy")
plt.xlabel("Sample Index")
plt.ylabel("Life Expectancy (Years)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest Performance:")
print("MAE:", mae_rf)
print("RMSE:", rmse_rf)
print("R²:", r2_rf)


In [ ]:
plt.figure(figsize=(8, 4))
indices = range(len(y_test))

plt.bar(indices, y_test, label="Actual", alpha=0.7)
plt.bar(indices, y_pred_rf, label="Predicted", alpha=0.7)

plt.title("Random Forest: Actual vs Predicted Life Expectancy")
plt.xlabel("Sample Index")
plt.ylabel("Life Expectancy (Years)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
gb = GradientBoostingRegressor(
    random_state=42
)
gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)

mae_gb = mean_absolute_error(y_test, y_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
r2_gb = r2_score(y_test, y_pred_gb)

print("Gradient Boosting Performance:")
print("MAE:", mae_gb)
print("RMSE:", rmse_gb)
print("R²:", r2_gb)


In [ ]:
plt.figure(figsize=(8, 4))
indices = range(len(y_test))

plt.bar(indices, y_test, label="Actual", alpha=0.7)
plt.bar(indices, y_pred_gb, label="Predicted", alpha=0.7)

plt.title("Gradient Boosting: Actual vs Predicted Life Expectancy")
plt.xlabel("Sample Index")
plt.ylabel("Life Expectancy (Years)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
importances = rf.feature_importances_
feature_names = X.columns

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

plt.figure(figsize=(8, 6))
sns.barplot(data=importance_df, x="Importance", y="Feature")
plt.title("Feature Importance from Random Forest")
plt.tight_layout()
plt.show()

importance_df
